In [61]:
import folium
from IPython.display import display

# --- paste your points here in [lat, lon] order ---
coords_latlon = [
    [49.018539, 8.299383],
    [49.018364, 8.300611],
    [49.017487, 8.300360],
    [49.016605, 8.300086],
    [49.015736, 8.299762],
    [49.015083, 8.298214],
    [49.015942, 8.298544],
    [49.016804, 8.298865],
    [49.017673, 8.299134],
    [49.018539, 8.299383],
    [49.097332, 8.359251],
    [49.096504, 8.358751],
    [49.095684, 8.358178],
    [49.094887, 8.357544],
    [49.094116, 8.356861],
    [49.019253, 8.300828],
    [49.018364, 8.300611],
    [49.018539, 8.299383],
    [49.019424, 8.299598],
    [49.020306, 8.299757],
    [49.095238, 8.356402],
    [49.096024, 8.357029],
    [49.096821, 8.357587],
    [49.097641, 8.358081],
    [49.097332, 8.359251],
]

# --- if your list is actually [lon, lat], flip it like this ---
# coords_latlon = [(lat, lon) for lon, lat in coords_lonlat]

# map bounds/center
lats = [p[0] for p in coords_latlon]
lons = [p[1] for p in coords_latlon]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

m = folium.Map(location=center, zoom_start=12,
         width="100%", height="600px")

colors = ["red", "blue", "yellow", "green", "black"]

segment = []
for i, (lat, lon) in enumerate(coords_latlon):
    color = colors[(i // 5) % len(colors)]
    # point
    folium.CircleMarker(
        location=(lat, lon),
        radius=4,
        color=color, fill=True, fill_opacity=0.9,
        tooltip=f"Pt {i+1} ({color})"
    ).add_to(m)

    # collect for 5-point segment polyline
    segment.append((lat, lon))
    if (i % 5 == 4) or (i == len(coords_latlon) - 1):
        folium.PolyLine(segment, color=color, weight=3, opacity=0.85).add_to(m)
        segment = []

m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
display(m)


In [62]:
import json
import folium
from IPython.display import display

# ---- set your file here ----
JSON_PATH = "rheinhafen bonding box.json"  # your file

# ---- helpers to extract coordinate sequences (as lists of [lon,lat]) ----
def extract_lines(obj):
    """
    Return a list of sequences, where each sequence is a list of [lon,lat] pairs.
    Handles FeatureCollection, Feature, LineString, MultiLineString, Polygon, MultiPolygon,
    or a plain dict with a top-level "coordinates".
    """
    if not isinstance(obj, dict):
        return []

    t = obj.get("type")
    if t == "FeatureCollection":
        lines = []
        for ft in obj.get("features", []):
            lines += extract_lines(ft)
        return lines
    if t == "Feature":
        return extract_lines(obj.get("geometry", {}))
    if t == "LineString":
        return [obj.get("coordinates", [])]
    if t == "MultiLineString":
        return obj.get("coordinates", [])
    if t == "Polygon":
        coords = obj.get("coordinates", [])
        return [coords[0]] if coords else []          # exterior ring as a line
    if t == "MultiPolygon":
        lines = []
        for poly in obj.get("coordinates", []):
            if poly: lines.append(poly[0])            # exterior rings
        return lines

    # Fallback: plain object with "coordinates": [[lon,lat],...]
    if "coordinates" in obj and isinstance(obj["coordinates"], list):
        coords = obj["coordinates"]
        if coords and isinstance(coords[0], list) and len(coords[0]) == 2 and isinstance(coords[0][0], (int,float)):
            return [coords]
    return []

# ---- load JSON and collect line sequences ([lon,lat]) ----
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

line_sequences_lonlat = extract_lines(data)
if not line_sequences_lonlat:
    raise ValueError("No coordinates found to plot.")

# ---- flip to [lat,lon] for Folium and compute bounds ----
all_latlons = []
sequences_latlon = []
for seq in line_sequences_lonlat:
    latlon_seq = [(lat, lon) for lon, lat in seq]  # flip here
    sequences_latlon.append(latlon_seq)
    all_latlons.extend(latlon_seq)

lats = [lt for lt, _ in all_latlons]
lons = [ln for _, ln in all_latlons]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

# ---- make map and plot ----
m = folium.Map(location=center, zoom_start=12, width="100%", height="600px")

# separate layers so you can toggle lines vs points
fg_lines = folium.FeatureGroup(name="Lines").add_to(m)
fg_points = folium.FeatureGroup(name="Vertices (click for coords)").add_to(m)

# add each linestring
for latlon_seq in sequences_latlon:
    folium.PolyLine(latlon_seq, color="blue", weight=3, opacity=0.85).add_to(fg_lines)
    # add a tiny marker at every vertex with popup + tooltip
    for i, (lat, lon) in enumerate(latlon_seq, start=1):
        folium.CircleMarker(
            location=(lat, lon),
            radius=2.5,
            color="#333",
            fill=True,
            fill_opacity=0.9,
            tooltip=f"Pt {i}: {lat:.6f}, {lon:.6f}",
            popup=folium.Popup(f"Lat: {lat:.6f}<br>Lon: {lon:.6f}", max_width=200),
        ).add_to(fg_points)

# click anywhere to see lat/lon
folium.LatLngPopup().add_to(m)

m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
folium.LayerControl().add_to(m)

display(m)


In [63]:
import json
import folium
from IPython.display import display

# --- set your file path here ---
JSON_PATH = "portlines.json"   # e.g. "rheinhafen bonding box.json"

# Load {"portline_1":[[lon,lat],...], ... "portline_6":[...]}
with open(JSON_PATH, "r", encoding="utf-8") as f:
    portlines = json.load(f)

# Sort keys so we draw 1..6 in order
keys_sorted = sorted(portlines.keys(), key=lambda k: int(k.split("_")[-1]))

# Convert to [lat,lon] for Folium and gather bounds
all_latlons = []
seqs_latlon = []
for k in keys_sorted:
    seq_lonlat = portlines[k]
    seq_latlon = [(lat, lon) for lon, lat in seq_lonlat]  # flip [lon,lat] -> [lat,lon]
    seqs_latlon.append((k, seq_latlon))
    all_latlons.extend(seq_latlon)

# Map center & bounds
lats = [lt for lt, _ in all_latlons]
lons = [ln for _, ln in all_latlons]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

# Make the map (set tiles if needed, e.g., tiles="CartoDB positron")
m = folium.Map(location=center, zoom_start=13, width="100%", height="600px")

# Colors for six lines
colors = ["red", "blue", "green", "purple", "orange", "black"]

# Add each portline with both polyline and point markers in the same layer
for idx, (name, latlon_seq) in enumerate(seqs_latlon):
    color = colors[idx % len(colors)]
    layer_name = name.replace("_", " ").title()  # "Portline 1"
    fg = folium.FeatureGroup(name=layer_name).add_to(m)

    # Polyline
    folium.PolyLine(
        latlon_seq,
        color=color,
        weight=4,
        opacity=0.9,
        tooltip=layer_name
    ).add_to(fg)

    # Vertex points (clickable)
    for i, (lat, lon) in enumerate(latlon_seq, start=1):
        folium.CircleMarker(
            location=(lat, lon),
            radius=3,
            color=color,
            fill=True,
            fill_opacity=0.9,
            tooltip=f"{layer_name} • Pt {i}",
            popup=folium.Popup(f"Lat: {lat:.6f}<br>Lon: {lon:.6f}", max_width=220),
        ).add_to(fg)

# Fit to data and show
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
folium.LayerControl().add_to(m)
folium.LatLngPopup().add_to(m)  # click anywhere to read coords
display(m)


In [64]:
import json
import folium
from IPython.display import display

# --- set your file path here ---
JSON_PATH = "polygons.json"   # e.g., "D:/path/to/polygons.json"

# --- load polygons ---
# Expected shape:
# {
#   "polygon_1": [[lon,lat], [lon,lat], ...],
#   "polygon_2": [...],
#   ...
# }
with open(JSON_PATH, "r", encoding="utf-8") as f:
    poly_dict = json.load(f)

# Make a stable order: polygon_1, polygon_2, ...
keys_sorted = sorted(poly_dict.keys(), key=lambda k: int(k.split("_")[-1]))

# Convert each polygon ring to [lat, lon] for Folium and collect bounds
polygons_latlon = []
all_latlons = []
for name in keys_sorted:
    ring_lonlat = poly_dict[name]                     # exterior ring only
    ring_latlon = [(lat, lon) for lon, lat in ring_lonlat]  # flip for Folium
    polygons_latlon.append((name, ring_latlon))
    all_latlons.extend(ring_latlon)

# Compute map center & bounds
lats = [lt for lt, _ in all_latlons]
lons = [ln for _, ln in all_latlons]
min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

# Create map (set a tile layer if your env needs it, e.g., tiles="CartoDB positron")
m = folium.Map(location=center, zoom_start=14, width="100%", height="600px")

# Colors (loops if > len(colors))
colors = ["red", "blue", "green", "purple", "orange", "black", "brown", "cadetblue"]

for idx, (name, ring_latlon) in enumerate(polygons_latlon):
    color = colors[idx % len(colors)]
    label = name.replace("_", " ").title()  # "Polygon 1"

    # Layer per polygon so you can toggle them
    layer = folium.FeatureGroup(name=label).add_to(m)

    # 1) Polygon shape
    folium.Polygon(
        locations=ring_latlon,
        color=color,
        weight=3,
        fill=True,
        fill_opacity=0.20,
        tooltip=label
    ).add_to(layer)

    # 2) Vertex points (clickable)
    for i, (lat, lon) in enumerate(ring_latlon, start=1):
        folium.CircleMarker(
            location=(lat, lon),
            radius=3,
            color=color,
            fill=True,
            fill_opacity=0.9,
            tooltip=f"{label} • Pt {i}",
            popup=folium.Popup(f"Lat: {lat:.6f}<br>Lon: {lon:.6f}", max_width=220),
        ).add_to(layer)

    # 3) Text label at polygon centroid (approx via vertex mean)
    c_lat = sum(lat for lat, _ in ring_latlon) / len(ring_latlon)
    c_lon = sum(lon for _, lon in ring_latlon) / len(ring_latlon)
    folium.Marker(
        location=(c_lat, c_lon),
        icon=folium.DivIcon(html=f"""<div style="font-size:12px;color:{color};
                                     background:rgba(255,255,255,0.6);
                                     padding:2px 4px;border-radius:4px;">
                                     {label}</div>""")
    ).add_to(layer)

# Fit to all polygons; add controls and click-to-read coordinates
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
folium.LayerControl().add_to(m)
folium.LatLngPopup().add_to(m)

display(m)


In [65]:
import json
import folium
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import substring, nearest_points
from IPython.display import display

# ---------------- CONFIG ----------------
POLYGONS_JSON   = "polygons.json"            # {"polygon_1":[[lon,lat],...], ... "polygon_5":[...]}
PORTLINES_JSON  = "portlines.json"           # {"portline_1":[[lon,lat],...], ..., "portline_6":[...]}
CENTERLINE_JSON = "sorted_centerline.json"   # [[lat,lon], ...]  (already lat,lon)

START_LATLON = (49.01047012778574, 8.333443504398304)
END_LATLON   = (49.03801668473199, 8.3037420358749)

# small numeric tolerances
ON_LINE_TOL = 2e-4     # ~20 m for "on the line" tests
LAT_EPS     = 1e-6     # latitude stop tolerance (~0.1 m)
# ---------------------------------------


# ---------- helpers ----------
def latlon_to_linestring(seq_latlon):
    """Convert [[lat,lon], ...] to shapely LineString (x=lon, y=lat)."""
    return LineString([(lon, lat) for (lat, lon) in seq_latlon])

def nearest_point_on_line(seq_latlon, ref_latlon):
    """Nearest point on a polyline (returned as [lat,lon])."""
    line = latlon_to_linestring(seq_latlon)
    p = Point(ref_latlon[1], ref_latlon[0])
    np = nearest_points(line, p)[0]
    return (np.y, np.x)

def segment_along_line(seq_latlon, a_latlon, b_latlon):
    """
    Forward-only subline from a to b (projections along the polyline),
    includes exact endpoints, de-duplicates.
    """
    line = latlon_to_linestring(seq_latlon)
    a_d  = line.project(Point(a_latlon[1], a_latlon[0]))
    b_d  = line.project(Point(b_latlon[1], b_latlon[0]))
    if a_d <= b_d:
        seg = substring(line, a_d, b_d)
    else:
        seg = substring(line, b_d, a_d)
        seg = LineString(list(seg.coords)[::-1])

    coords = [(y, x) for (x, y) in seg.coords]
    if not coords or coords[0] != a_latlon: coords = [a_latlon] + coords
    if not coords or coords[-1] != b_latlon: coords = coords + [b_latlon]
    out = []
    for p in coords:
        if not out or p != out[-1]:
            out.append(p)
    return out

def pick_prev_centerline_vertex(centerline_latlon, join_latlon, end_latlon):
    """
    Return the 'previous' centerline vertex relative to motion from join→END,
    guaranteeing we pick an interior vertex (not the endpoint).
    """
    cl_line = latlon_to_linestring(centerline_latlon)
    L = cl_line.length
    dj = cl_line.project(Point(join_latlon[1], join_latlon[0]))
    de = cl_line.project(Point(end_latlon[1],  end_latlon[0]))

    # keep de strictly inside (avoid snapping to endpoints)
    eps = 1e-9 * L if L > 0 else 0.0
    if de <= eps:     de = min(eps*10, L/1000.0)
    if de >= L - eps: de = max(L - eps*10, L - L/1000.0)

    forward = de >= dj
    cl_dists = [cl_line.project(Point(pt[1], pt[0])) for pt in centerline_latlon]

    prev_idx = None
    for i in range(len(cl_dists) - 1):
        a, b = cl_dists[i], cl_dists[i+1]
        if a <= de <= b:
            prev_idx = i if forward else i+1
            break
        if b <= de <= a:
            prev_idx = i+1 if forward else i
            break

    if prev_idx is None:
        prev_idx = len(centerline_latlon) - 2 if forward else 1
    return centerline_latlon[prev_idx]

def splice_pl6_to_centerline(pl6_latlon, centerline_latlon, join_on_pl6):
    """
    Find a good splice between portline_6 and the centerline.
    Prefer true intersections; otherwise use nearest points between the two polylines.
    Returns (p6_splice_latlon, cl_splice_latlon).
    """
    pl6_line = latlon_to_linestring(pl6_latlon)
    cl_line  = latlon_to_linestring(centerline_latlon)
    inter    = pl6_line.intersection(cl_line)
    join_d   = pl6_line.project(Point(join_on_pl6[1], join_on_pl6[0]))

    candidates_on_pl6 = []

    def add_candidate(pt):
        if pt.is_empty: return
        if isinstance(pt, Point):
            candidates_on_pl6.append((pt.y, pt.x))
        elif hasattr(pt, "geoms"):  # MultiPoint/GeometryCollection
            for g in pt.geoms:
                add_candidate(g)
        elif isinstance(pt, LineString):
            coords = list(pt.coords)
            if coords:
                candidates_on_pl6.append((coords[0][1], coords[0][0]))
                candidates_on_pl6.append((coords[-1][1], coords[-1][0]))

    add_candidate(inter)

    if candidates_on_pl6:
        # first intersection reached when moving along pl6 from the join
        def forward_dist(latlon):
            d = pl6_line.project(Point(latlon[1], latlon[0]))
            return d - join_d if d >= join_d else float("inf")
        p6_splice = min(candidates_on_pl6, key=forward_dist)
        if forward_dist(p6_splice) == float("inf"):
            q6, qc = nearest_points(pl6_line, cl_line)
            return (q6.y, q6.x), (qc.y, qc.x)
        qc = nearest_points(cl_line, Point(p6_splice[1], p6_splice[0]))[0]
        return p6_splice, (qc.y, qc.x)
    else:
        q6, qc = nearest_points(pl6_line, cl_line)
        return (q6.y, q6.x), (qc.y, qc.x)

def trim_centerline_by_end_lat(seg_latlon, end_lat):
    """
    Given a centerline segment seg_latlon ([lat,lon] from start→end),
    stop BEFORE we pass END's latitude. Used only when finishing via centerline.
    """
    if not seg_latlon:
        return seg_latlon

    lat0, lat1 = seg_latlon[0][0], seg_latlon[-1][0]
    lat_increasing = (lat1 >= lat0)

    out = []
    for pt in seg_latlon:
        lat = pt[0]
        passed = (lat_increasing and lat >  end_lat + LAT_EPS) or \
                 (not lat_increasing and lat < end_lat - LAT_EPS)
        out.append(pt)
        if passed:
            out.pop()  # drop the point that crossed the latitude
            break
        if abs(lat - end_lat) <= LAT_EPS:
            break

    if not out:
        out = [seg_latlon[0]]
    return out


# ---------- load data ----------
with open(POLYGONS_JSON, "r", encoding="utf-8") as f:
    polys_lonlat = json.load(f)     # [lon,lat]
with open(PORTLINES_JSON, "r", encoding="utf-8") as f:
    ports_lonlat = json.load(f)     # [lon,lat]
with open(CENTERLINE_JSON, "r", encoding="utf-8") as f:
    centerline_latlon = json.load(f)  # [lat,lon]

# Convert portlines to [lat,lon]
ports_latlon = {k: [(lat, lon) for (lon, lat) in seq] for k, seq in ports_lonlat.items()}

# ---------- choose portline_N from START polygon ----------
p_start = Point(START_LATLON[1], START_LATLON[0])
containing, nearest_name, nearest_d = None, None, float("inf")
for name in sorted(polys_lonlat.keys(), key=lambda k: int(k.split("_")[-1])):
    poly = Polygon(polys_lonlat[name])  # still [lon,lat]
    if poly.contains(p_start) or poly.touches(p_start):
        containing = name
        break
    d = poly.exterior.distance(p_start)
    if d < nearest_d:
        nearest_d, nearest_name = d, name
if containing is None:
    containing = nearest_name
N = int(containing.split("_")[-1])
print(f"Start is in/near: {containing}  → using portline_{N}")

# ---------- START → along portline_N to its junction with portline_6 ----------
plN = ports_latlon[f"portline_{N}"]
pl6 = ports_latlon["portline_6"]
if not plN or not pl6:
    raise RuntimeError("Missing portline_N or portline_6")

# choose plN endpoint closer to pl6 as the junction end
pl6_line = latlon_to_linestring(pl6)
endA, endB = plN[0], plN[-1]
junc_end = endA if pl6_line.distance(Point(endA[1], endA[0])) <= pl6_line.distance(Point(endB[1], endB[0])) else endB

start_on_plN = nearest_point_on_line(plN, START_LATLON)
plN_segment   = segment_along_line(plN, start_on_plN, junc_end)

# exact join point on pl6
join_on_pl6   = nearest_point_on_line(pl6, junc_end)

# ---------- along portline_6 to a good splice with the centerline ----------
p6_splice, cl_splice = splice_pl6_to_centerline(pl6, centerline_latlon, join_on_pl6)
pl6_leg = segment_along_line(pl6, join_on_pl6, p6_splice)

# ---------- centerline to the 'previous' vertex, trimmed by END latitude, then straight to END ----------
prev_point = pick_prev_centerline_vertex(centerline_latlon, cl_splice, END_LATLON)
cl_leg     = segment_along_line(centerline_latlon, cl_splice, prev_point)
cl_leg     = trim_centerline_by_end_lat(cl_leg, END_LATLON[0])  # << NEW trim-by-lat rule

last_on_cl = cl_leg[-1] if cl_leg else cl_splice
final_leg  = [last_on_cl, END_LATLON]

# ---------- assemble final route ----------
route = [START_LATLON] + plN_segment + pl6_leg[1:] + cl_leg[1:] + final_leg

# ---------- plot ----------
all_pts = route[:]
for ring in polys_lonlat.values():
    all_pts += [(lat, lon) for (lon, lat) in ring]
min_lat = min(p[0] for p in all_pts); max_lat = max(p[0] for p in all_pts)
min_lon = min(p[1] for p in all_pts); max_lon = max(p[1] for p in all_pts)
center  = [(min_lat + max_lat)/2, (min_lon + max_lon)/2]

m = folium.Map(location=center, zoom_start=13, width="100%", height="600px")  # no tiles arg

# context (optional)
poly_layer = folium.FeatureGroup(name="Polygons").add_to(m)
colors = ["red","blue","green","purple","orange"]
for i,name in enumerate(sorted(polys_lonlat.keys(), key=lambda k: int(k.split("_")[-1]))):
    ring_latlon = [(lat,lon) for (lon,lat) in polys_lonlat[name]]
    folium.Polygon(ring_latlon, color=colors[i%len(colors)], weight=2, fill=False,
                   tooltip=name.replace("_"," ").title()).add_to(poly_layer)

folium.PolyLine(pl6, color="blue",  weight=3, opacity=0.35, tooltip="portline_6 (context)").add_to(m)
folium.PolyLine(centerline_latlon, color="gray", weight=2, opacity=0.6, tooltip="centerline (context)").add_to(m)

# final route
folium.PolyLine(route, color="black", weight=6, opacity=0.95, tooltip="Route (plN → pl6 → centerline → END)").add_to(m)
folium.Marker(START_LATLON, icon=folium.Icon(color="green"), popup="START").add_to(m)
folium.Marker(END_LATLON,   icon=folium.Icon(color="red"),   popup="END").add_to(m)

folium.LayerControl().add_to(m)
folium.LatLngPopup().add_to(m)
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
display(m)


Start is in/near: polygon_4  → using portline_4


In [73]:
import json
import folium
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import substring, nearest_points
from IPython.display import display

# ---------------- CONFIG ----------------
POLYGONS_JSON   = "polygons.json"            # {"polygon_1":[[lon,lat],...], ... "polygon_5":[...]}
PORTLINES_JSON  = "portlines.json"           # {"portline_1":[[lon,lat],...], ..., "portline_6":[...]}
CENTERLINE_JSON = "sorted_centerline.json"   # [[lat,lon], ...]  (already lat,lon)

START_LATLON = (49.012970559008686, 8.341085965875727)
END_LATLON   = (49.03801668473199, 8.3037420358749)

# small numeric tolerances
ON_LINE_TOL = 2e-4     # ~20 m for "on the line" tests
LAT_EPS     = 1e-6     # latitude stop tolerance (~0.1 m)
# ---------------------------------------


# ---------- helpers ----------
def latlon_to_linestring(seq_latlon):
    """Convert [[lat,lon], ...] to shapely LineString (x=lon, y=lat)."""
    return LineString([(lon, lat) for (lat, lon) in seq_latlon])

def nearest_point_on_line(seq_latlon, ref_latlon):
    """Nearest point on a polyline (returned as [lat,lon])."""
    line = latlon_to_linestring(seq_latlon)
    p = Point(ref_latlon[1], ref_latlon[0])
    np = nearest_points(line, p)[0]
    return (np.y, np.x)

def segment_along_line(seq_latlon, a_latlon, b_latlon):
    """
    Forward-only subline from a to b (projections along the polyline),
    includes exact endpoints, de-duplicates.
    """
    line = latlon_to_linestring(seq_latlon)
    a_d  = line.project(Point(a_latlon[1], a_latlon[0]))
    b_d  = line.project(Point(b_latlon[1], b_latlon[0]))
    if a_d <= b_d:
        seg = substring(line, a_d, b_d)
    else:
        seg = substring(line, b_d, a_d)
        seg = LineString(list(seg.coords)[::-1])

    coords = [(y, x) for (x, y) in seg.coords]
    if not coords or coords[0] != a_latlon: coords = [a_latlon] + coords
    if not coords or coords[-1] != b_latlon: coords = coords + [b_latlon]
    out = []
    for p in coords:
        if not out or p != out[-1]:
            out.append(p)
    return out

def pick_prev_centerline_vertex(centerline_latlon, join_latlon, end_latlon):
    """
    Return the 'previous' centerline vertex relative to motion from join→END,
    guaranteeing we pick an interior vertex (not the endpoint).
    """
    cl_line = latlon_to_linestring(centerline_latlon)
    L = cl_line.length
    dj = cl_line.project(Point(join_latlon[1], join_latlon[0]))
    de = cl_line.project(Point(end_latlon[1],  end_latlon[0]))

    # keep de strictly inside (avoid snapping to endpoints)
    eps = 1e-9 * L if L > 0 else 0.0
    if de <= eps:     de = min(eps*10, L/1000.0)
    if de >= L - eps: de = max(L - eps*10, L - L/1000.0)

    forward = de >= dj
    cl_dists = [cl_line.project(Point(pt[1], pt[0])) for pt in centerline_latlon]

    prev_idx = None
    for i in range(len(cl_dists) - 1):
        a, b = cl_dists[i], cl_dists[i+1]
        if a <= de <= b:
            prev_idx = i if forward else i+1
            break
        if b <= de <= a:
            prev_idx = i+1 if forward else i
            break

    if prev_idx is None:
        prev_idx = len(centerline_latlon) - 2 if forward else 1
    return centerline_latlon[prev_idx]

def splice_pl6_to_centerline(pl6_latlon, centerline_latlon, join_on_pl6):
    """
    Find a good splice between portline_6 and the centerline.
    Prefer true intersections; otherwise use nearest points between the two polylines.
    Returns (p6_splice_latlon, cl_splice_latlon).
    """
    pl6_line = latlon_to_linestring(pl6_latlon)
    cl_line  = latlon_to_linestring(centerline_latlon)
    inter    = pl6_line.intersection(cl_line)
    join_d   = pl6_line.project(Point(join_on_pl6[1], join_on_pl6[0]))

    candidates_on_pl6 = []

    def add_candidate(pt):
        if pt.is_empty: return
        if isinstance(pt, Point):
            candidates_on_pl6.append((pt.y, pt.x))
        elif hasattr(pt, "geoms"):  # MultiPoint/GeometryCollection
            for g in pt.geoms:
                add_candidate(g)
        elif isinstance(pt, LineString):
            coords = list(pt.coords)
            if coords:
                candidates_on_pl6.append((coords[0][1], coords[0][0]))
                candidates_on_pl6.append((coords[-1][1], coords[-1][0]))

    add_candidate(inter)

    if candidates_on_pl6:
        # first intersection reached when moving along pl6 from the join
        def forward_dist(latlon):
            d = pl6_line.project(Point(latlon[1], latlon[0]))
            return d - join_d if d >= join_d else float("inf")
        p6_splice = min(candidates_on_pl6, key=forward_dist)
        if forward_dist(p6_splice) == float("inf"):
            q6, qc = nearest_points(pl6_line, cl_line)
            return (q6.y, q6.x), (qc.y, qc.x)
        qc = nearest_points(cl_line, Point(p6_splice[1], p6_splice[0]))[0]
        return p6_splice, (qc.y, qc.x)
    else:
        q6, qc = nearest_points(pl6_line, cl_line)
        return (q6.y, q6.x), (qc.y, qc.x)

def trim_centerline_by_end_lat(seg_latlon, end_lat):
    """
    Given a centerline segment seg_latlon ([lat,lon] from start→end),
    stop BEFORE we pass END's latitude. Used only when finishing via centerline.
    """
    if not seg_latlon:
        return seg_latlon

    lat0, lat1 = seg_latlon[0][0], seg_latlon[-1][0]
    lat_increasing = (lat1 >= lat0)

    out = []
    for pt in seg_latlon:
        lat = pt[0]
        passed = (lat_increasing and lat >  end_lat + LAT_EPS) or \
                 (not lat_increasing and lat < end_lat - LAT_EPS)
        out.append(pt)
        if passed:
            out.pop()  # drop the point that crossed the latitude
            break
        if abs(lat - end_lat) <= LAT_EPS:
            break

    if not out:
        out = [seg_latlon[0]]
    return out


# ---------- load data ----------
with open(POLYGONS_JSON, "r", encoding="utf-8") as f:
    polys_lonlat = json.load(f)     # [lon,lat]
with open(PORTLINES_JSON, "r", encoding="utf-8") as f:
    ports_lonlat = json.load(f)     # [lon,lat]
with open(CENTERLINE_JSON, "r", encoding="utf-8") as f:
    centerline_latlon = json.load(f)  # [lat,lon]

# Convert portlines to [lat,lon]
ports_latlon = {k: [(lat, lon) for (lon, lat) in seq] for k, seq in ports_lonlat.items()}

# ---------- choose portline_N from START polygon ----------
p_start = Point(START_LATLON[1], START_LATLON[0])
containing, nearest_name, nearest_d = None, None, float("inf")
for name in sorted(polys_lonlat.keys(), key=lambda k: int(k.split("_")[-1])):
    poly = Polygon(polys_lonlat[name])  # still [lon,lat]
    if poly.contains(p_start) or poly.touches(p_start):
        containing = name
        break
    d = poly.exterior.distance(p_start)
    if d < nearest_d:
        nearest_d, nearest_name = d, name
if containing is None:
    containing = nearest_name
N = int(containing.split("_")[-1])
print(f"Start is in/near: {containing}  → using portline_{N}")

# ---------- START → along portline_N to its junction with portline_6 ----------
plN = ports_latlon[f"portline_{N}"]
pl6 = ports_latlon["portline_6"]
if not plN or not pl6:
    raise RuntimeError("Missing portline_N or portline_6")

# choose plN endpoint closer to pl6 as the junction end
pl6_line = latlon_to_linestring(pl6)
endA, endB = plN[0], plN[-1]
junc_end = endA if pl6_line.distance(Point(endA[1], endA[0])) <= pl6_line.distance(Point(endB[1], endB[0])) else endB

start_on_plN = nearest_point_on_line(plN, START_LATLON)
plN_segment   = segment_along_line(plN, start_on_plN, junc_end)
plN_segment.remove(plN_segment[0])  # remove the first point (START_LATLON)

# exact join point on pl6
join_on_pl6   = nearest_point_on_line(pl6, junc_end)

# ---------- along portline_6 to a good splice with the centerline ----------
p6_splice, cl_splice = splice_pl6_to_centerline(pl6, centerline_latlon, join_on_pl6)
pl6_leg = segment_along_line(pl6, join_on_pl6, p6_splice)

# ---------- centerline to the 'previous' vertex, trimmed by END latitude, then straight to END ----------
prev_point = pick_prev_centerline_vertex(centerline_latlon, cl_splice, END_LATLON)
cl_leg     = segment_along_line(centerline_latlon, cl_splice, prev_point)
cl_leg     = trim_centerline_by_end_lat(cl_leg, END_LATLON[0])  # << trim-by-lat rule

last_on_cl = cl_leg[-1] if cl_leg else cl_splice
final_leg  = [last_on_cl, END_LATLON]

# ---------- assemble final route ----------
route = [START_LATLON] + plN_segment + pl6_leg[1:] + cl_leg[1:] + final_leg

# ---------- plot ----------
all_pts = route[:]
min_lat = min(p[0] for p in all_pts); max_lat = max(p[0] for p in all_pts)
min_lon = min(p[1] for p in all_pts); max_lon = max(p[1] for p in all_pts)
center  = [(min_lat + max_lat)/2, (min_lon + max_lon)/2]

m = folium.Map(location=center, zoom_start=13, width="100%", height="600px")  # no tiles arg

# (Polygons REMOVED from map)

# optional context (keep these or remove as you like)
folium.PolyLine(pl6, color="blue",  weight=3, opacity=0.35, tooltip="portline_6 (context)").add_to(m)
folium.PolyLine(centerline_latlon, color="gray", weight=2, opacity=0.6, tooltip="centerline (context)").add_to(m)

# final route
folium.PolyLine(route, color="black", weight=6, opacity=0.95, tooltip="Route (plN → pl6 → centerline → END)").add_to(m)
folium.Marker(START_LATLON, icon=folium.Icon(color="green"), popup="START").add_to(m)
folium.Marker(END_LATLON,   icon=folium.Icon(color="red"),   popup="END").add_to(m)

folium.LayerControl().add_to(m)
folium.LatLngPopup().add_to(m)
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
display(m)


Start is in/near: polygon_3  → using portline_3


### Helper

In [67]:
def latlon_to_linestring(seq_latlon):
    """Convert [[lat,lon], ...] to shapely LineString (x=lon, y=lat)."""
    return LineString([(lon, lat) for (lat, lon) in seq_latlon])

def nearest_point_on_line(seq_latlon, ref_latlon):
    """Nearest point on a polyline (returned as [lat,lon])."""
    line = latlon_to_linestring(seq_latlon)
    p = Point(ref_latlon[1], ref_latlon[0])
    np = nearest_points(line, p)[0]
    return (np.y, np.x)

def segment_along_line(seq_latlon, a_latlon, b_latlon):
    """
    Forward-only subline from a to b (projections along the polyline),
    includes exact endpoints, de-duplicates.
    """
    line = latlon_to_linestring(seq_latlon)
    a_d  = line.project(Point(a_latlon[1], a_latlon[0]))
    b_d  = line.project(Point(b_latlon[1], b_latlon[0]))
    if a_d <= b_d:
        seg = substring(line, a_d, b_d)
    else:
        seg = substring(line, b_d, a_d)
        seg = LineString(list(seg.coords)[::-1])

    coords = [(y, x) for (x, y) in seg.coords]
    if not coords or coords[0] != a_latlon: coords = [a_latlon] + coords
    if not coords or coords[-1] != b_latlon: coords = coords + [b_latlon]
    out = []
    for p in coords:
        if not out or p != out[-1]:
            out.append(p)
    return out

def pick_prev_centerline_vertex(centerline_latlon, join_latlon, end_latlon):
    """
    Return the 'previous' centerline vertex relative to motion from join→END,
    guaranteeing we pick an interior vertex (not the endpoint).
    """
    cl_line = latlon_to_linestring(centerline_latlon)
    L = cl_line.length
    dj = cl_line.project(Point(join_latlon[1], join_latlon[0]))
    de = cl_line.project(Point(end_latlon[1],  end_latlon[0]))

    # keep de strictly inside (avoid snapping to endpoints)
    eps = 1e-9 * L if L > 0 else 0.0
    if de <= eps:     de = min(eps*10, L/1000.0)
    if de >= L - eps: de = max(L - eps*10, L - L/1000.0)

    forward = de >= dj
    cl_dists = [cl_line.project(Point(pt[1], pt[0])) for pt in centerline_latlon]

    prev_idx = None
    for i in range(len(cl_dists) - 1):
        a, b = cl_dists[i], cl_dists[i+1]
        if a <= de <= b:
            prev_idx = i if forward else i+1
            break
        if b <= de <= a:
            prev_idx = i+1 if forward else i
            break

    if prev_idx is None:
        prev_idx = len(centerline_latlon) - 2 if forward else 1
    return centerline_latlon[prev_idx]

def splice_pl6_to_centerline(pl6_latlon, centerline_latlon, join_on_pl6):
    """
    Find a good splice between portline_6 and the centerline.
    Prefer true intersections; otherwise use nearest points between the two polylines.
    Returns (p6_splice_latlon, cl_splice_latlon).
    """
    pl6_line = latlon_to_linestring(pl6_latlon)
    cl_line  = latlon_to_linestring(centerline_latlon)
    inter    = pl6_line.intersection(cl_line)
    join_d   = pl6_line.project(Point(join_on_pl6[1], join_on_pl6[0]))

    candidates_on_pl6 = []

    def add_candidate(pt):
        if pt.is_empty: return
        if isinstance(pt, Point):
            candidates_on_pl6.append((pt.y, pt.x))
        elif hasattr(pt, "geoms"):  # MultiPoint/GeometryCollection
            for g in pt.geoms:
                add_candidate(g)
        elif isinstance(pt, LineString):
            coords = list(pt.coords)
            if coords:
                candidates_on_pl6.append((coords[0][1], coords[0][0]))
                candidates_on_pl6.append((coords[-1][1], coords[-1][0]))

    add_candidate(inter)

    if candidates_on_pl6:
        # first intersection reached when moving along pl6 from the join
        def forward_dist(latlon):
            d = pl6_line.project(Point(latlon[1], latlon[0]))
            return d - join_d if d >= join_d else float("inf")
        p6_splice = min(candidates_on_pl6, key=forward_dist)
        if forward_dist(p6_splice) == float("inf"):
            q6, qc = nearest_points(pl6_line, cl_line)
            return (q6.y, q6.x), (qc.y, qc.x)
        qc = nearest_points(cl_line, Point(p6_splice[1], p6_splice[0]))[0]
        return p6_splice, (qc.y, qc.x)
    else:
        q6, qc = nearest_points(pl6_line, cl_line)
        return (q6.y, q6.x), (qc.y, qc.x)

def trim_centerline_by_end_lat(seg_latlon, end_lat):
    """
    Given a centerline segment seg_latlon ([lat,lon] from start→end),
    stop BEFORE we pass END's latitude. Used only when finishing via centerline.
    """
    if not seg_latlon:
        return seg_latlon

    lat0, lat1 = seg_latlon[0][0], seg_latlon[-1][0]
    lat_increasing = (lat1 >= lat0)

    out = []
    for pt in seg_latlon:
        lat = pt[0]
        passed = (lat_increasing and lat >  end_lat + LAT_EPS) or \
                 (not lat_increasing and lat < end_lat - LAT_EPS)
        out.append(pt)
        if passed:
            out.pop()  # drop the point that crossed the latitude
            break
        if abs(lat - end_lat) <= LAT_EPS:
            break

    if not out:
        out = [seg_latlon[0]]
    return out

In [68]:
import json
import folium
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import substring, nearest_points
from IPython.display import display

# ---------------- CONFIG ----------------
POLYGONS_JSON   = "polygons.json"            # {"polygon_1":[[lon,lat],...], ... "polygon_5":[...]}
PORTLINES_JSON  = "portlines.json"           # {"portline_1":[[lon,lat],...], ..., "portline_6":[...]}
CENTERLINE_JSON = "sorted_centerline.json"   # [[lat,lon], ...]  (already lat,lon)



# small numeric tolerances
ON_LINE_TOL = 2e-4     # ~20 m for "on the line" tests
LAT_EPS     = 1e-6     # latitude stop tolerance (~0.1 m)
# ---------------------------------------


# ---------- helpers ----------



# ---------- load data ----------
with open(POLYGONS_JSON, "r", encoding="utf-8") as f:
    polys_lonlat = json.load(f)     # [lon,lat]
with open(PORTLINES_JSON, "r", encoding="utf-8") as f:
    ports_lonlat = json.load(f)     # [lon,lat]
with open(CENTERLINE_JSON, "r", encoding="utf-8") as f:
    centerline_latlon = json.load(f)  # [lat,lon]

# Convert portlines to [lat,lon]
ports_latlon = {k: [(lat, lon) for (lon, lat) in seq] for k, seq in ports_lonlat.items()}

In [69]:
def lineofsight(START_LATLON, END_LATLON,polys_lonlat, ports_latlon, centerline_latlon):
    # ---------- choose portline_N from START polygon ----------
    p_start = Point(START_LATLON[1], START_LATLON[0])
    containing, nearest_name, nearest_d = None, None, float("inf")
    for name in sorted(polys_lonlat.keys(), key=lambda k: int(k.split("_")[-1])):
        poly = Polygon(polys_lonlat[name])  # still [lon,lat]
        if poly.contains(p_start) or poly.touches(p_start):
            containing = name
            break
        d = poly.exterior.distance(p_start)
        if d < nearest_d:
            nearest_d, nearest_name = d, name
    if containing is None:
        containing = nearest_name
    N = int(containing.split("_")[-1])
    print(f"Start is in/near: {containing}  → using portline_{N}")

    # ---------- START → along portline_N to its junction with portline_6 ----------
    plN = ports_latlon[f"portline_{N}"]
    pl6 = ports_latlon["portline_6"]
    if not plN or not pl6:
        raise RuntimeError("Missing portline_N or portline_6")

    # choose plN endpoint closer to pl6 as the junction end
    pl6_line = latlon_to_linestring(pl6)
    endA, endB = plN[0], plN[-1]
    junc_end = endA if pl6_line.distance(Point(endA[1], endA[0])) <= pl6_line.distance(Point(endB[1], endB[0])) else endB

    start_on_plN = nearest_point_on_line(plN, START_LATLON)
    plN_segment   = segment_along_line(plN, start_on_plN, junc_end)

    # exact join point on pl6
    join_on_pl6   = nearest_point_on_line(pl6, junc_end)

    # ---------- along portline_6 to a good splice with the centerline ----------
    p6_splice, cl_splice = splice_pl6_to_centerline(pl6, centerline_latlon, join_on_pl6)
    pl6_leg = segment_along_line(pl6, join_on_pl6, p6_splice)

    # ---------- centerline to the 'previous' vertex, trimmed by END latitude, then straight to END ----------
    prev_point = pick_prev_centerline_vertex(centerline_latlon, cl_splice, END_LATLON)
    cl_leg     = segment_along_line(centerline_latlon, cl_splice, prev_point)
    cl_leg     = trim_centerline_by_end_lat(cl_leg, END_LATLON[0])  # << trim-by-lat rule

    last_on_cl = cl_leg[-1] if cl_leg else cl_splice
    final_leg  = [last_on_cl, END_LATLON]

    # ---------- assemble final route ----------
    route = [START_LATLON] + plN_segment + pl6_leg[1:] + cl_leg[1:] + final_leg
    return route

In [70]:
START_LATLON = (49.012970559008686, 8.341085965875727)
END_LATLON   = (49.00850695851945, 8.29468555110602)

line = lineofsight(START_LATLON, END_LATLON, polys_lonlat, ports_latlon, centerline_latlon)
print("Line of sight route:", line)

Start is in/near: polygon_3  → using portline_3
Line of sight route: [(49.012970559008686, 8.341085965875727), (49.01292581232742, 8.341084968802583), (49.0130732, 8.3344705), (49.0146312, 8.3313811), (49.0156399, 8.3287443), (49.0158233, 8.3218964), (49.0160438, 8.3136334), (49.0161057, 8.3083965), (49.0163176, 8.3036861), (49.0162602, 8.3024454), (49.0165457, 8.3013368), (49.0175405, 8.2997866), (49.016714500000006, 8.299451999999999), (49.015844, 8.299150000000001), (49.0149805, 8.2988155), (49.0141335, 8.2984665), (49.013262, 8.298098), (49.012434, 8.297692000000001), (49.01157, 8.297254500000001), (49.0107425, 8.296747), (49.0099295, 8.296202000000001), (49.0091345, 8.295583), (49.0091345, 8.295583), (49.00850695851945, 8.29468555110602)]


In [71]:
# ---------- plot ----------
all_pts = line[:]
min_lat = min(p[0] for p in all_pts); max_lat = max(p[0] for p in all_pts)
min_lon = min(p[1] for p in all_pts); max_lon = max(p[1] for p in all_pts)
center  = [(min_lat + max_lat)/2, (min_lon + max_lon)/2]

m = folium.Map(location=center, zoom_start=13, width="100%", height="600px")  # no tiles arg

# (Polygons REMOVED from map)

# optional context (keep these or remove as you like)
folium.PolyLine(pl6, color="blue",  weight=3, opacity=0.35, tooltip="portline_6 (context)").add_to(m)
folium.PolyLine(centerline_latlon, color="gray", weight=2, opacity=0.6, tooltip="centerline (context)").add_to(m)

# final route
folium.PolyLine(line, color="black", weight=6, opacity=0.95, tooltip="Route (plN → pl6 → centerline → END)").add_to(m)
folium.Marker(START_LATLON, icon=folium.Icon(color="green"), popup="START").add_to(m)
folium.Marker(END_LATLON,   icon=folium.Icon(color="red"),   popup="END").add_to(m)

folium.LayerControl().add_to(m)
folium.LatLngPopup().add_to(m)
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
display(m)

In [82]:
def calculate_bearing(pointA, pointB):
    """
    Returns bearing from pointA to pointB (in degrees)
    """
    lat1, lon1 = map(math.radians, pointA)
    lat2, lon2 = map(math.radians, pointB)

    d_lon = lon2 - lon1

    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(d_lon)

    bearing = math.atan2(x, y)
    return (90-math.degrees(bearing) + 360) % 360

start = (49.0129207, 8.3413144)   # (lat, lon)
p1    = (49.0130732, 8.3344705)   # (lat, lon)

print(calculate_bearing(p1, start))   # ≈ 91.94° (east)
print(calculate_bearing(start, p1))   # ≈ 271.95° (west)

358.0568097090934
178.05164353380678


In [83]:
from geopy.distance import geodesic

start = (49.0129207, 8.3413144)   # (lat, lon)
p1    = (49.0130732, 8.3344705)   # (lat, lon)

d_m  = geodesic(start, p1).meters
d_km = geodesic(start, p1).kilometers
d_nm = d_km / 1.852  # nautical miles

print(f"Distance: {d_m:.1f} m ({d_km:.3f} km, {d_nm:.3f} nmi)")


Distance: 500.9 m (0.501 km, 0.270 nmi)
